<a href="https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print(hf_token is not None, len(hf_token) if hf_token else 0)  # sanity check

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"

True 37


In [3]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

q_agg = f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions,
       SUM(gsc_clicks) AS clicks,
       SUM(gsc_sum_position) AS sum_position
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-03'
GROUP BY 1,2
HAVING SUM(gsc_impressions) > 0
"""
page_month = con.execute(q_agg).df()
page_month['avg_position'] = page_month['sum_position'] / page_month['impressions']
page_month['ctr'] = page_month['clicks'] / page_month['impressions']

import pandas as pd
bins = [0, 3, 10, 20, 50, float('inf')]
labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
page_month['position_tier'] = pd.cut(page_month['avg_position'], bins=bins, labels=labels)

bucket1 = page_month.groupby('position_tier').agg(
    mean_ctr=('ctr', 'mean'),
    n=('ctr', 'size')
)
print(bucket1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

               mean_ctr      n
position_tier                 
top_3          0.009961  17426
page_1         0.004873  83288
striking       0.003285  29922
page_3_5       0.002379  32240
deep           0.000846  12428


/tmp/ipykernel_3208/512949015.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = page_month.groupby('position_tier').agg(


In [4]:
q_content = f"SELECT content_hash_id, content_updated_date FROM read_parquet('{BASE}/dim_content.parquet')"
content_dates = con.execute(q_content).df()

merged = page_month.merge(content_dates, on='content_hash_id', how='left')
merged['content_updated_date'] = pd.to_datetime(merged['content_updated_date'])
merged['days_since_update'] = (pd.Timestamp('2026-03-31') - merged['content_updated_date']).dt.days

stale_bins = [0, 30, 90, 180, 365, float('inf')]
stale_labels = ['<30d', '30-90d', '90-180d', '180-365d', '365d+']
merged['staleness_bucket'] = pd.cut(merged['days_since_update'], bins=stale_bins, labels=stale_labels)

bucket2 = merged.groupby('staleness_bucket').agg(
    mean_ctr=('ctr', 'mean'),
    mean_impressions=('impressions', 'mean'),
    n=('ctr', 'size')
)
print(bucket2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                  mean_ctr  mean_impressions      n
staleness_bucket                                   
<30d              0.003959        145.918398    674
30-90d            0.002194       1319.644653  25696
90-180d           0.020107        359.545660   1325
180-365d          0.002208         66.670498    261
365d+                  NaN               NaN      0


/tmp/ipykernel_3208/2148971647.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2 = merged.groupby('staleness_bucket').agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
tier_benchmark = page_month.groupby('position_tier')['ctr'].transform('mean')
merged2 = page_month.copy()
merged2['expected_ctr_for_tier'] = tier_benchmark
merged2['ctr_gap'] = merged2['expected_ctr_for_tier'] - merged2['ctr']

# score: weight the CTR gap by visibility, so bigger opportunities (high impressions) rank higher
merged2['score'] = merged2['ctr_gap'] * merged2['impressions']
merged2['reason_code'] = 'CTR_GAP_VS_POSITION_TIER'
merged2['action'] = merged2['score'].apply(lambda s: 'review_for_refresh' if s > 0 else 'no_action')

ranked = merged2.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(ranked.head(20)[['client_hash_id','content_hash_id','impressions','ctr','avg_position','position_tier','expected_ctr_for_tier','ctr_gap','score','reason_code','action']])


/tmp/ipykernel_3208/1461937161.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_benchmark = page_month.groupby('position_tier')['ctr'].transform('mean')


             client_hash_id           content_hash_id  impressions       ctr  \
0   client_23a62021009f63c4  content_44f34c0a90047651     212404.0  0.000113   
1   client_e547b89c05043229  content_8d7d99f109e19aa2     203497.0  0.001420   
2   client_e547b89c05043229  content_0e03de7680314cd5     221310.0  0.003253   
3   client_73cda7b4e4f265ea  content_8e1334d6356668e3     134984.0  0.000007   
4   client_e547b89c05043229  content_4ffe18112a5642e3     186983.0  0.003134   
5   client_73cda7b4e4f265ea  content_fec55986a1868d62     124075.0  0.000008   
6   client_e547b89c05043229  content_ec2e0346994fb5a5     245276.0  0.006034   
7   client_e547b89c05043229  content_545bb6cc7081ded3     122905.0  0.002335   
8   client_73cda7b4e4f265ea  content_9c057b66c30a3abb      83834.0  0.000012   
9   client_e547b89c05043229  content_9ef3d7516483e665      89229.0  0.001031   
10  client_e547b89c05043229  content_306bc78dff1eb683      80821.0  0.000433   
11  client_e547b89c05043229  content_c46

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# content_44f34c0a90047651 (top_3, 212K impr, CTR 0.011%) — flagged for near-zero CTR despite huge visibility; wrong if this impression volume is inflated/non-human traffic rather than real search demand.
#content_8d7d99f109e19aa2 (top_3, 203K impr) — same pattern, second-highest score; wrong if client_e547b89c05043229 has an impression-counting quirk affecting all its pages equally.
#content_0e03de7680314cd5 (top_3, 221K impr) — same client again; wrong if this is really one systemic client issue counted as three separate "opportunities."
#content_8e1334d6356668e3 (top_3, 135K impr, CTR 0.0007%) — near-zero actual clicks; wrong if the true intent behind these impressions isn't clickable (e.g. answer-box satisfied queries).
#content_4ffe18112a5642e3 (top_3, 187K impr) — consistent with cluster pattern; wrong if refreshing content won't move CTR when the real issue is query-intent mismatch, not content quality.
#content_fec55986a1868d62 (top_3, 124K impr, CTR 0.0008%) — same client_73cda7b4e4f265ea cluster; wrong if this client's whole account has a tracking issue rather than page-level content problems.
#content_ec2e0346994fb5a5 (top_3, 245K impr, highest impressions in top 20) — flagged for scale; wrong if this page is a high-volume evergreen page where low CTR is expected (e.g. an FAQ that's usually skimmed, not clicked).
#content_545bb6cc7081ded3 (top_3, 123K impr) — same client cluster; wrong if refresh has already happened recently and this data predates it.
#content_9c057b66c30a3abb (top_3, 84K impr, CTR 0.0012%) — extremely low CTR; wrong if this page's ranking keyword doesn't match its actual content (a targeting problem, not a freshness one).
#content_9ef3d7516483e665 (top_3, 89K impr) — same top cluster; wrong if seasonal/temporary query spikes are inflating this month's impressions specifically.
#content_306bc78dff1eb683 (top_3, 81K impr) — lower down the list but same pattern; wrong if this content type structurally gets lower CTR regardless of quality (e.g. a reference page vs. a buying-intent page).
#content_c46df0fa61530d86 (top_3, 70K impr) — same client, smaller scale; wrong if it's already been refreshed once this quarter and this is stale scoring.
#content_34a70fea29d15f24 (page_1, 143K impr, avg_position 3.17) — first non-top_3 entry; wrong if page_1-tier expectations (0.49% CTR) are themselves too optimistic for this query type.
#content_80eb6221de550658 (top_3, 80K impr) — back in the main cluster; wrong if refreshing won't help because the underlying keyword is genuinely low-intent.
#content_6a9c79f55413b447 (top_3, 73K impr) — same; wrong for the same intent-mismatch reason as above.
#content_b13e95d379c78818 (top_3, 76K impr) — smaller client cluster; wrong if client_62f4a7e64f5e0096's traffic source differs structurally (e.g. mostly branded queries with different CTR norms).
#content_e0ca055423cbe896 (top_3, 86K impr) — same; wrong for same reason.
#content_b2b85c287474668d (top_3, 65K impr) — lowest-impression top_3 entry in the list; wrong if this is a borderline case where noise, not real signal, pushed it into the top 20.
#content_252aa5480bb1f8d7 (top_3, 67K impr) — same client cluster as row 6/9/14/15; wrong if it's the same systemic issue repeated, inflating this client's representation in the "top opportunities" unfairly.
#content_b99ea6861864dea5 (page_1, 194K impr, avg_position 4.55) — largest page_1 entry; wrong if page-1-tier is simply a noisier bucket than top_3 and this page is actually fine for its type.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# leakage sanity check confirm no future-window or product-flag columns snuck in
print(ranked.columns.tolist())


['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'sum_position', 'avg_position', 'ctr', 'position_tier', 'expected_ctr_for_tier', 'ctr_gap', 'score', 'reason_code', 'action']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.